In [ ]:
import csv
import os

# --- Configuration ---
# Folder where the input CSV is located and output CSV will be saved
DATA_FOLDER = 'data'
# Name of the input CSV file (ensure this file exists in the DATA_FOLDER)
INPUT_CSV_FILENAME = 'train_data.csv'
# Name for the output CSV file
OUTPUT_CSV_FILENAME = 'output_sampled_1000.csv'

# Number of lines from the input file to consider as the source pool for selection
# This script assumes we are selecting from the first TOTAL_LINES_IN_SOURCE_POOL data lines.
TOTAL_LINES_IN_SOURCE_POOL = 6983
# Number of lines to select from the source pool
LINES_TO_EXTRACT = 3000
# --- End of Configuration ---

def sample_csv_lines(input_filepath, output_filepath, total_source_lines, lines_to_sample):
    """
    Reads an input CSV, samples a specified number of lines from a defined source pool,
    and writes the sampled lines to an output CSV.

    Args:
        input_filepath (str): Path to the input CSV file.
        output_filepath (str): Path to save the output CSV file.
        total_source_lines (int): The number of data lines from the start of the file
                                  to consider as the pool for sampling.
        lines_to_sample (int): The number of lines to sample from the pool.
    """
    try:
        # Ensure the output directory exists
        os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

        all_rows = []
        header = []

        with open(input_filepath, 'r', newline='', encoding='utf-8') as infile:
            reader = csv.reader(infile)
            try:
                header = next(reader)  # Read the header row
            except StopIteration:
                print(f"Error: Input CSV file '{input_filepath}' is empty or has no header.")
                return False

            for row in reader:
                all_rows.append(row) # Read all data rows

        # Check if there are enough data rows to form the source pool
        if len(all_rows) < total_source_lines:
            print(f"Error: Input CSV '{input_filepath}' has {len(all_rows)} data lines, "
                  f"which is less than the required {total_source_lines} lines for the source pool.")
            print("Please ensure the input file has enough data.")
            return False

        # Define the actual pool of rows to sample from
        source_rows_for_selection = all_rows[:total_source_lines]

        if lines_to_sample < 0:
            print("Error: Number of lines to extract cannot be negative.")
            return False
        if lines_to_sample > total_source_lines:
            print(f"Warning: Requesting to sample {lines_to_sample} lines, but source pool is only "
                  f"{total_source_lines} lines. All {total_source_lines} lines will be selected.")
            lines_to_sample = total_source_lines # Adjust to select all available if over-requested

        selected_indices = []
        if lines_to_sample == 0:
            pass # No lines to select
        elif lines_to_sample == 1:
            if total_source_lines > 0:
                selected_indices = [0] # Select the first line of the pool
            # If total_source_lines is 0, selected_indices remains empty, which is correct.
        elif total_source_lines > 0 : # lines_to_sample > 1 and total_source_lines > 0
            # Generate N indices spread across a range of M items (0 to M-1)
            # The formula j * (M-1)/(N-1) for j in 0..N-1 gives N points
            # that include the start (0) and end (M-1) of the range.
            # Rounding these points gives the nearest integer indices.
            # This ensures the lines are as "equally spaced" as possible.
            # M = total_source_lines
            # N = lines_to_sample
            # Denominator (N-1) cannot be zero here because lines_to_sample > 1
            denominator = lines_to_sample - 1
            # Numerator base (M-1)
            # If total_source_lines is 1, (total_source_lines - 1) is 0.
            # This case is fine: round(j * 0 / denominator) = 0. All indices will be 0.
            # However, this specific path is taken if lines_to_sample > 1.
            # If total_source_lines is 1 and lines_to_sample > 1, it's an impossible selection
            # (handled by the warning/adjustment above).
            # So, here, total_source_lines >= lines_to_sample > 1, meaning total_source_lines > 1.
            # Thus, (total_source_lines - 1) >= 0.

            for j in range(lines_to_sample):
                # Calculate the floating point index
                float_idx = j * (total_source_lines - 1) / denominator
                # Round to the nearest whole number for the actual index
                selected_indices.append(int(round(float_idx)))

        # Remove potential duplicate indices if lines_to_sample is very close to total_source_lines
        # and rounding causes collisions, then sort. For M=6983, N=1000, duplicates are highly unlikely.
        # If exactly N unique lines are critical even in edge cases, a more complex selection might be needed,
        # but for the given numbers, this should yield N unique indices.
        if len(set(selected_indices)) < lines_to_sample and lines_to_sample > 0 and total_source_lines > 0:
            print(f"Warning: Due to rounding, {len(set(selected_indices))} unique lines were selected instead of {lines_to_sample}.")
            # Fallback to unique sorted indices if duplicates were somehow generated
            selected_indices = sorted(list(set(selected_indices)))


        output_rows = [source_rows_for_selection[i] for i in selected_indices if i < len(source_rows_for_selection)]

        with open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            if header: # Write header if it was read
                writer.writerow(header)
            writer.writerows(output_rows)

        print(f"Successfully sampled {len(output_rows)} lines from '{input_filepath}'.")
        print(f"Output saved to '{output_filepath}'.")
        return True

    except FileNotFoundError:
        print(f"Error: Input file '{input_filepath}' not found.")
        return False
    except IOError as e:
        print(f"An I/O error occurred: {e}")
        return False
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return False

if __name__ == "__main__":
    # Construct full paths
    # Assumes the script is run from the parent directory of DATA_FOLDER
    # e.g., if script is in /home/user/project/ and data is in /home/user/project/data/
    input_file = os.path.join(DATA_FOLDER, INPUT_CSV_FILENAME)
    output_file = os.path.join(DATA_FOLDER, OUTPUT_CSV_FILENAME)

    print(f"Starting CSV sampling process...")
    print(f"Input file: {input_file}")
    print(f"Output file: {output_file}")
    print(f"Source pool size: First {TOTAL_LINES_IN_SOURCE_POOL} data lines.")
    print(f"Number of lines to extract: {LINES_TO_EXTRACT}.")

    # Create the data directory if it doesn't exist (primarily for output, but good practice)
    if not os.path.exists(DATA_FOLDER):
        try:
            os.makedirs(DATA_FOLDER)
            print(f"Created data directory: '{DATA_FOLDER}'")
        except OSError as e:
            print(f"Error creating data directory '{DATA_FOLDER}': {e}")
            # Exit if we can't create the directory where output should go
            exit()


    sample_csv_lines(input_file, output_file, TOTAL_LINES_IN_SOURCE_POOL, LINES_TO_EXTRACT)

Starting CSV sampling process...
Input file: data/train_data.csv
Output file: data/output_sampled_1000.csv
Source pool size: First 6983 data lines.
Number of lines to extract: 3000.
Successfully sampled 3000 lines from 'data/train_data.csv'.
Output saved to 'data/output_sampled_1000.csv'.
